# Telecom Plan Recommendation: Classification Model

A mobile operator wants a model that analyzes subscribers' usage behavior and recommends the more suitable of two plans. The target variable `is_ultra` indicates whether the subscriber uses the Ultra plan (1) or not (0).

**Objective:** build a classification model that reaches an accuracy of at least **0.75** (stored as `acc_limit`) on held-out data, comparing Random Forest, Logistic Regression and Decision Tree classifiers.

## Setup

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

## Load Data

In [16]:
df = pd.read_csv('./datasets/users_behavior.csv')

## Data Overview

The dataset has 3,214 rows and no missing values. Each row describes one subscriber's usage behavior: number of calls (`calls`), total call duration in minutes (`minutes`), number of text messages (`messages`) and internet traffic in MB (`mb_used`), plus the target `is_ultra`. All features are numeric, so no encoding is required.

In [19]:
df.info()
df.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


## Train / Validation Split

The data is split into a training set (75%) and a validation set (25%) with a fixed random seed. The validation set is used both to tune hyperparameters and to compare models.

In [22]:
df_train, df_valid = train_test_split(df, test_size=0.25, random_state=54321)

features_train = df_train.drop(['is_ultra'], axis=1)
target_train = df_train['is_ultra']

features_valid = df_valid.drop(['is_ultra'],axis=1)
target_valid = df_valid['is_ultra']

acc_limit = 0.75

## Model Comparison

### Random Forest

The number of trees (`n_estimators`) is tuned from 1 to 19, keeping the value with the highest validation accuracy.

In [ ]:
## Random Forest Classifier

best_score = 0
best_est = 0
for est in range(1, 20):
    model = RandomForestClassifier(random_state=54321, n_estimators=est)
    model.fit(features_train, target_train) # train on the training set
    score = model.score(features_valid,target_valid) # accuracy on the validation set
    if score > best_score:
        best_score = score
        best_est = est

print("MODELO UTILIZADO - Random Forest Classifier")
print("Acurácia do melhor modelo no conjunto de validação (n_estimators = {}): {}".format(best_est, best_score))

MODELO UTILIZADO = Random Forest Classifier
Acurácia do melhor modelo no conjunto de validação (n_estimators = 16): 0.8022388059701493


### Logistic Regression

A Logistic Regression model (`liblinear` solver) is trained as a linear baseline; accuracy is reported on both the training and validation sets.

In [ ]:
## Logistic Regression

model = LogisticRegression(random_state=54321, solver='liblinear')
model.fit(features_train,target_train)
score_train = model.score(features_train,target_train)
score_valid = model.score(features_valid,target_valid)

print("MODELO UTILIZADO - Logistic Regression")
print("Acurácia do modelo de regressão logística no conjunto de treinamento:", score_train)
print("Acurácia do modelo de regressão logística no conjunto de validação:", score_valid)

Acurácia do modelo de regressão logística no conjunto de treinamento: 0.7178423236514523
Acurácia do modelo de regressão logística no conjunto de validação: 0.6940298507462687


### Decision Tree

Tree depths (`max_depth`) from 1 to 19 are evaluated on the validation set.

> **Note:** inside this loop, `best_acc` and `best_depth` are reset on every iteration, so the printed result corresponds to the last depth evaluated (`max_depth = 19`) rather than to the best-performing depth.

In [ ]:
## Decision Tree Classifier

for depth in range(1, 20):
        model = DecisionTreeClassifier(random_state=12345, max_depth = depth)

        model.fit(features_train,target_train)

        predictions_valid = model.predict(features_valid)

        best_acc = 0
        best_depth = 0
        x = accuracy_score(target_valid, predictions_valid)
        if x > best_acc:
                best_acc = x
                best_depth = depth
print("Acurácia do melhor modelo no conjunto de validação:{})\n profundidade = {}".format(best_acc, best_depth))

Acurácia do melhor modelo no conjunto de validação:0.7313432835820896)
 profundidade = 19


## Conclusions

| Model | Configuration | Validation accuracy |
|---|---|---|
| Random Forest | `n_estimators = 16` (best of 1–19) | **0.802** |
| Decision Tree | `max_depth = 19` (last value in the loop) | 0.731 |
| Logistic Regression | `solver = 'liblinear'` | 0.694 (training: 0.718) |

- The Random Forest classifier is the only model that exceeds the 0.75 accuracy threshold, reaching 0.802 on the validation set with 16 trees.
- Logistic Regression performs below the threshold on both the training and validation sets, which suggests that a linear decision boundary is not flexible enough for this problem.
- The Random Forest is therefore the recommended model. Because the same validation set was used for tuning and for model comparison, an additional held-out test set would be needed to obtain an unbiased estimate of its final accuracy.